# 2. Importación de Datos Históricos

Importar y consolidar datos históricos desde múltiples fuentes en `data/raw/`.

## Objetivo
- Leer archivos CSV existentes
- Validar y normalizar esquemas
- Consolidar en dataset único
- Guardar en `data/processed/`

In [7]:
# ============================================================================
# IMPORTAR LIBRERÍAS
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Configuración
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
sns.set_palette('husl')

# Rutas
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

print('✓ Librerías cargadas')

✓ Librerías cargadas


## Cargar Datos Disponibles

In [2]:
# ============================================================================
# BUSCAR Y CARGAR ARCHIVOS CSV
# ============================================================================
csv_files = sorted(DATA_RAW.glob('*.csv'))
print(f'Archivos encontrados: {len(csv_files)}')

dataframes = {}
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        dataframes[csv_file.stem] = df
        print(f'  ✓ {csv_file.name}: {len(df):,} registros, {len(df.columns)} columnas')
    except Exception as e:
        print(f'  ✗ Error: {csv_file.name} - {e}')

if not dataframes:
    print('  ⚠ No hay archivos CSV disponibles')

Archivos encontrados: 5
  ✓ crowdsourcing_aggregated.csv: 148 registros, 9 columnas
  ✓ crowdsourcing_raw.csv: 500 registros, 11 columnas
  ✗ Error: scraped_traffic.csv - No columns to parse from file
  ✓ synthetic_traffic.csv: 5,040 registros, 8 columnas
  ✓ traffic_data.csv: 18 registros, 5 columnas


## Funciones de Validación

In [3]:
# ============================================================================
# FUNCIONES DE VALIDACIÓN
# ============================================================================
def normalize_columns(df):
    """
    Normalizar nombres de columnas:
    - Minúsculas
    - Sin espacios
    - Sin caracteres especiales
    """
    df.columns = (df.columns
                  .str.lower()
                  .str.strip()
                  .str.replace(' ', '_')
                  .str.replace('[^a-z0-9_]', '', regex=True))
    return df

def validate_schema(df, required_cols):
    """
    Verificar que DataFrame tenga columnas requeridas.
    Retorna (valid: bool, missing: set)
    """
    current_cols = set(df.columns)
    required_set = set(required_cols)
    missing = required_set - current_cols
    return len(missing) == 0, missing

def check_data_quality(df, name):
    """
    Resumen de calidad de datos.
    """
    print(f'\n{name}:')
    print(f'  Dimensiones: {df.shape}')
    print(f'  Valores nulos: {df.isnull().sum().sum()}')
    print(f'  Duplicados: {df.duplicated().sum()}')
    print(f'  Tipos: {dict(df.dtypes)}')

## Normalizar Datos

In [4]:
# ============================================================================
# NORMALIZAR ESQUEMAS
# ============================================================================
if dataframes:
    REQUIRED_COLS = ['timestamp', 'velocity', 'density', 'flow']
    
    print('\nNormalizando columnas...')
    for name in list(dataframes.keys()):
        dataframes[name] = normalize_columns(dataframes[name])
        valid, missing = validate_schema(dataframes[name], REQUIRED_COLS)
        status = '✓' if valid else f'✗ Faltan: {missing}'
        print(f'  {name}: {status}')
        
        # Mostrar calidad
        check_data_quality(dataframes[name], name)


Normalizando columnas...
  crowdsourcing_aggregated: ✗ Faltan: {'velocity', 'density', 'flow'}

crowdsourcing_aggregated:
  Dimensiones: (148, 9)
  Valores nulos: 15
  Duplicados: 0
  Tipos: {'timestamp': <StringDtype(storage='python', na_value=nan)>, 'road': <StringDtype(storage='python', na_value=nan)>, 'avg_speed': dtype('float64'), 'std_speed': dtype('float64'), 'min_speed': dtype('float64'), 'max_speed': dtype('float64'), 'avg_confidence': dtype('float64'), 'congestion_reports': dtype('int64'), 'unique_users': dtype('int64')}
  crowdsourcing_raw: ✗ Faltan: {'velocity', 'density', 'flow'}

crowdsourcing_raw:
  Dimensiones: (500, 11)
  Valores nulos: 0
  Duplicados: 0
  Tipos: {'user_id': <StringDtype(storage='python', na_value=nan)>, 'timestamp': <StringDtype(storage='python', na_value=nan)>, 'road': <StringDtype(storage='python', na_value=nan)>, 'latitude': dtype('float64'), 'longitude': dtype('float64'), 'speed_kmh': dtype('float64'), 'report_type': <StringDtype(storage='python'

## Consolidar Datos

In [5]:
# ============================================================================
# CONSOLIDAR
# ============================================================================
if dataframes:
    df_consolidated = pd.concat(
        dataframes.values(),
        ignore_index=True,
        sort=False
    )
    
    print(f'\n✓ Consolidación completa:')
    print(f'  Total registros: {len(df_consolidated):,}')
    print(f'  Columnas: {list(df_consolidated.columns)}')
    
    # Eliminar duplicados exactos
    duplicates_before = len(df_consolidated)
    df_consolidated = df_consolidated.drop_duplicates()
    duplicates_removed = duplicates_before - len(df_consolidated)
    print(f'  Duplicados removidos: {duplicates_removed}')
else:
    print('No hay datos para consolidar')


✓ Consolidación completa:
  Total registros: 5,706
  Columnas: ['timestamp', 'road', 'avg_speed', 'std_speed', 'min_speed', 'max_speed', 'avg_confidence', 'congestion_reports', 'unique_users', 'user_id', 'latitude', 'longitude', 'speed_kmh', 'report_type', 'confidence', 'device_type', 'app_version', 'hour', 'velocity_kmh', 'density_veh_km', 'flow_veh_h', 'wait_time_sec', 'stops_count', 'avenida', 'velocidad', 'densidad', 'detenciones']
  Duplicados removidos: 0


## Exportar

In [6]:
# ============================================================================
# EXPORTAR
# ============================================================================
if 'df_consolidated' in locals():
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    
    output_file = DATA_PROCESSED / 'historical_consolidated.csv'
    df_consolidated.to_csv(output_file, index=False)
    
    print(f'✓ Exportación completa:')
    print(f'  Archivo: {output_file}')
    print(f'  Tamaño: {df_consolidated.memory_usage(deep=True).sum() / 1024:.2f} KB')

✓ Exportación completa:
  Archivo: ..\data\processed\historical_consolidated.csv
  Tamaño: 2652.40 KB
